# Cycle 7 : Approche finale


In [111]:
### Import des modules
%load_ext autoreload
%autoreload 2

import joblib
import pandas as pd
import numpy as np
from models import utils
from models.technova_features import TechNovaFeatureEngineering
from models.technova_correlation_cleaning import CorrelationFilter
import sklearn
from sklearn.model_selection import FixedThresholdClassifier, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, recall_score, f1_score, roc_auc_score, classification_report, make_scorer, fbeta_score
from scipy.stats import uniform, loguniform
import json, datetime

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [96]:
data = pd.read_csv('../data/rafined/employees.csv', sep=',')

In [97]:
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

In [98]:
# Encodage des variables catégorielles et standardisation des variables numériques
numeric_features = data.select_dtypes(include=['int64', 'float64'])
categorical_features = ['statut_marital', 'departement', 'poste', 'domaine_etude']

preprocessor = ColumnTransformer(
    transformers=[
        ('standard_scaler', StandardScaler(), make_column_selector(dtype_include='number')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'), make_column_selector(dtype_include='object')),
    ],
    remainder='drop'
)

In [99]:
nb_features_to_keep = 15

In [100]:
# Permet de ne garder que nb_feature_to_keep pour réduire le bruit
feature_selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    max_features=nb_features_to_keep,
    threshold=-np.inf
)

In [109]:
# Benchmark du modele avec CV
logistic_regression_model = LogisticRegression(
    random_state=42,
    max_iter=5000,
    class_weight='balanced',
    solver='saga'
)

threshold_model = FixedThresholdClassifier(
    estimator=logistic_regression_model,
    threshold=0.5,
    response_method="predict_proba"
)

pipeline = Pipeline([
    ('features', TechNovaFeatureEngineering()),
    ('corr_cleaning', CorrelationFilter(threshold=0.80)),
    ('preprocessor', preprocessor),
    ('feature_selection', feature_selector),
    ('model', threshold_model),
])

utils.benchmark(pipeline, train_data)

--- Validation Fold Results ---
Validation Recall : [0.65789474 0.84210526 0.68421053 0.65789474 0.81578947], Recall moyen : 0.7315789473684211
Validation F1-Scores [0.43478261 0.47058824 0.47706422 0.4587156  0.55357143], F1 moyen : 0.4789444178149919
Validation ROC AUC : [0.69790005 0.84517766 0.81204916 0.75474219 0.85372696], ROC moyen : 0.7927192037932087

--- Train Fold Results (Overfit Check) ---
Train Recall : [0.79605263 0.72368421 0.78289474 0.73684211 0.71052632], Recall moyen : 0.75
Train F1-Scores [0.53658537 0.47722343 0.52422907 0.4838013  0.46551724], F1 moyen : 0.4974712810702105
Train ROC AUC : [0.85244957 0.81296278 0.82771329 0.82380261 0.80198119], ROC moyen : 0.8237818896921908


In [108]:
# Recherche des meilleurs hyper-paramètres
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

full_pipeline = Pipeline([
    ('features', TechNovaFeatureEngineering()),
    ('corr_cleaning', CorrelationFilter(threshold=0.80)),
    ('preprocessor', preprocessor),
    ('feature_selection', feature_selector),
    ('model', threshold_model)
])

param_distributions = {
    'model__estimator__C': loguniform(1e-4, 1e2),
    'model__estimator__l1_ratio': uniform(0, 1),
    'model__threshold': [0.3, 0.4, 0.5, 0.6],
}

f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)

search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring=f2_scorer,
    cv=5,
    random_state=42,
    n_jobs=-1,
)

search.fit(train_data['X_train'], train_data['y_train'])

print(f"Meilleurs paramètres : {search.best_params_}")
print(f"f1 moyen en Validation Croisée : {search.best_score_:.4f}\n")

best_pipeline = search.best_estimator_

Meilleurs paramètres : {'model__estimator__C': np.float64(0.010051981180656781), 'model__estimator__l1_ratio': np.float64(0.14286681792194078), 'model__threshold': 0.5}
f1 moyen en Validation Croisée : 0.6073



In [103]:
# Affichage des performances du modèle
utils.benchmark(best_pipeline, train_data)

y_pred_test = best_pipeline.predict(train_data['X_test'])
y_probs_test = best_pipeline.predict_proba(train_data['X_test'])[:, 1]
y_pred_test = best_pipeline.predict(train_data['X_test'])

y_probs_train = best_pipeline.predict_proba(train_data['X_train'])[:, 1]
y_pred_train = best_pipeline.predict(train_data['X_train'])

auc_train = roc_auc_score(train_data['y_train'], y_probs_train)
auc_test = roc_auc_score(train_data['y_test'], y_probs_test)

recall_train = recall_score(train_data['y_train'], y_pred_train)
recall_test = recall_score(train_data['y_test'], y_pred_test)

f1_train = f1_score(train_data['y_train'], y_pred_train)
f1_test = f1_score(train_data['y_test'], y_pred_test)

print("Vérif overfit")
print(f"ROC AUC - Train Set : {auc_train:.4f}")
print(f"ROC AUC - Test Set  : {auc_test:.4f}")
print(f"Différence ROC AUC (Overfit) : {auc_train - auc_test:.4f}\n")
print(f"Recall - Train Set : {recall_train:.4f}")
print(f"Recall - Test Set  : {recall_test:.4f}")
print(f"Différence Recall (Overfit) : {recall_train - recall_test:.4f}\n")
print(f"f1 - Train Set : {f1_train:.4f}")
print(f"f1 - Test Set  : {f1_test:.4f}")
print(f"Différence f1 (Overfit) : {f1_train - f1_test:.4f}\n")

print("Performances")
print(classification_report(train_data['y_test'], y_pred_test))

print("Matrice de Confusion :")
print(confusion_matrix(train_data['y_test'], y_pred_test))

--- Validation Fold Results ---
Validation Recall : [0.63157895 0.84210526 0.73684211 0.60526316 0.84210526], Recall moyen : 0.731578947368421
Validation F1-Scores [0.44036697 0.48484848 0.50909091 0.42592593 0.56637168], F1 moyen : 0.4853207947516626
Validation ROC AUC : [0.6920521  0.84985306 0.8232701  0.74993321 0.85586428], ROC moyen : 0.7941945503660705

--- Train Fold Results (Overfit Check) ---
Train Recall : [0.77631579 0.70394737 0.75       0.71710526 0.71052632], Recall moyen : 0.7315789473684211
Train F1-Scores [0.52212389 0.46521739 0.51006711 0.47807018 0.46753247], F1 moyen : 0.4886022084349363
Train ROC AUC : [0.84779923 0.80028017 0.81744046 0.81664832 0.79472684], ROC moyen : 0.8153790032056459
Vérif overfit
ROC AUC - Train Set : 0.8186
ROC AUC - Test Set  : 0.7670
Différence ROC AUC (Overfit) : 0.0516

Recall - Train Set : 0.7263
Recall - Test Set  : 0.7234
Différence Recall (Overfit) : 0.0029

f1 - Train Set : 0.4868
f1 - Test Set  : 0.4658
Différence f1 (Overfit) :

In [104]:
# Entrainement final avec 100% des données + les parametres trouvé par RandomizedSearchCV
best_params = search.best_params_

X_complet = data.drop('a_quitte_l_entreprise', axis=1)
y_complet = data['a_quitte_l_entreprise']

production_pipeline = Pipeline([
    ('features', TechNovaFeatureEngineering()),
    ('corr_cleaning', CorrelationFilter(threshold=0.80)),
    ('preprocessor', preprocessor),
    ('feature_selection', feature_selector),
    ('model', FixedThresholdClassifier(
        estimator=LogisticRegression(
            random_state=42,
            class_weight='balanced',
            max_iter=5000,
            solver='saga'
        ),
        threshold=0.5,
        response_method="predict_proba"
    ))
])

production_pipeline.set_params(**best_params)

production_pipeline.fit(X_complet, y_complet)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('corr_cleaning', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
Name,Type,Value
quartiles_par_poste_,dict,"{'As...on': array([2386. ...87.5, 3902.5]), 'Ca...al': array([5025.2.... , 8538.75]), 'Co...nt': array([2379.5...86. , 3880.5]), 'Di...ue': array([13490.... , 18814.5 ]), ...}"
,threshold,0.8
Name,Type,Value
columns_to_drop_,list,"['ni...te', 'sc...rt', 'sc...ce', 'ev...ce']"


In [112]:
#Export des modèles pour la production...

from pathlib import Path
import os

base_dir = Path(os.getcwd()).resolve().parent

if not os.path.exists(base_dir / 'models/compiled/'):
    os.mkdir(base_dir / 'models/compiled/')

version = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

metadata = {
    "version": version,
    "n_features_out": nb_features_to_keep,
    "target": "a_quitte_l_entreprise",
    "sklearn_version": sklearn.__version__,
}

final_preprocessor = Pipeline(production_pipeline.steps[:-1])
final_model = production_pipeline.named_steps['model']

joblib.dump(final_preprocessor, base_dir / 'models/compiled/classification_preprocessor.joblib')
joblib.dump(final_model, base_dir / 'models/compiled/classification_model.joblib')

with open(base_dir / f'models/compiled/metadata_{version}.json', 'w') as f:
    json.dump(metadata, f, indent=2)